# 02 - Embeddings y base vectorial (Chroma)

En este cuaderno convertimos cada trozo de texto en un vector de números (embedding) y los guardamos en una base vectorial, Chroma. Así el asistente podrá buscar los trozos cuyo significado se parece más a una pregunta, aunque no usen las mismas palabras.

# Importaciones

- **pandas**: para cargar los trozos.
- **sentence_transformers**: descarga y usa el modelo que convierte texto en embeddings.
- **chromadb**: la base de datos donde guardamos los embeddings y hacemos las búsquedas.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

# Carga de los trozos

Cargamos los trozos que guardamos en el cuaderno 01.

In [2]:
df = pd.read_csv('../data/processed/chunks.csv')
df.head()

,fuente,texto
0,rd_1457_1986_talleres_reparacion.pdf,"Real Decreto 1457/1986, de 10 de enero, por el..."
1,rd_1457_1986_talleres_reparacion.pdf,................ 4\nArtículo 3. Clasificación ...
2,rd_1457_1986_talleres_reparacion.pdf,..................... 7\nTÍTULO III. Centros d...
3,rd_1457_1986_talleres_reparacion.pdf,a. ........................................ 10...
4,rd_1457_1986_talleres_reparacion.pdf,............... 13\nDisposición transitoria se...


In [3]:
df.shape

(850, 2)

In [4]:
df['fuente'].value_counts()

fuente
rd_920_2017_inspeccion_tecnica_vehiculos.html    287
rd_920_2017_inspeccion_tecnica_vehiculos.xml     275
registro_estaciones_itv_2026.csv                 143
rd_1457_1986_talleres_reparacion.pdf              76
dudas_anexo_ix_reglamento_vehiculos.pdf           51
dgt_llamadas_a_revision.html                      10
faq_taller.md                                      8
Name: count, dtype: int64

# Modelo de embeddings

Usamos **intfloat/multilingual-e5-small**, un modelo gratuito y ligero entrenado en muchos idiomas, incluido el español, que da buenos resultados en búsqueda de documentos. Lo elegimos frente a paraphrase-multilingual-MiniLM-L12-v2 porque este último solo lee 128 tokens y cortaría nuestros trozos de 800 caracteres, mientras que e5-small lee hasta 512.

Este modelo pide añadir delante la palabra passage: a los documentos y query: a las preguntas, para distinguir unos de otras.

In [5]:
model = SentenceTransformer('intfloat/multilingual-e5-small')
model.max_seq_length

512

Comprobamos cuántos tokens (trozos de palabra que lee el modelo) ocupa nuestro trozo más largo, para asegurarnos de que cabe en ese máximo.

In [6]:
tokens = []
for texto in df['texto']:
    tokens.append(len(model.tokenizer.tokenize('passage: ' + texto)))

print('Máximo de tokens en un trozo:', max(tokens))

Máximo de tokens en un trozo: 360


# Cálculo de los embeddings

Añadimos passage: delante de cada trozo y calculamos sus embeddings. normalize_embeddings=True deja todos los vectores del mismo tamaño, lo que mejora la comparación entre ellos.

In [7]:
textos = []
for texto in df['texto']:
    textos.append('passage: ' + texto)

embeddings = model.encode(textos, normalize_embeddings=True, show_progress_bar=False)
embeddings.shape

(850, 384)

Cada trozo es ahora un vector de 384 números.

# Base vectorial con Chroma

PersistentClient guarda la base en la carpeta chroma_db, así no hay que recalcular los embeddings cada vez. Si la colección taller ya existe la borramos, para que el cuaderno se pueda ejecutar de nuevo desde cero. Con hnsw:space = cosine le indicamos que mida la similitud por coseno.

In [8]:
client = chromadb.PersistentClient(path='../chroma_db')

nombres = []
for c in client.list_collections():
    nombres.append(c.name)
if 'taller' in nombres:
    client.delete_collection('taller')

collection = client.create_collection(name='taller', metadata={'hnsw:space': 'cosine'})

Preparamos lo que pide Chroma para cada trozo: un id único, el texto, su embedding y un metadato con la fuente (el archivo del que sale).

In [9]:
ids = []
metadatos = []
for i in range(len(df)):
    ids.append('trozo_' + str(i))
    metadatos.append({'fuente': df['fuente'][i]})

documentos = list(df['texto'])
vectores = embeddings.tolist()

Los añadimos por lotes de 500 para no mandar demasiados datos de golpe.

In [10]:
lote = 500
for inicio in range(0, len(ids), lote):
    fin = inicio + lote
    collection.add(ids=ids[inicio:fin], documents=documentos[inicio:fin],
                   embeddings=vectores[inicio:fin], metadatas=metadatos[inicio:fin])

collection.count()

850

# Búsquedas de prueba

Para buscar, convertimos la pregunta en embedding (con query: delante) y pedimos a Chroma los 3 trozos más parecidos. Mostramos la fuente, la distancia (cuanto más baja, más parecido) y el principio del texto.

In [11]:
pregunta = '¿Cada cuánto tengo que pasar la ITV?'
vector = model.encode('query: ' + pregunta, normalize_embeddings=True).tolist()
resultados = collection.query(query_embeddings=[vector], n_results=3)

for i in range(3):
    print(resultados['metadatas'][0][i]['fuente'], round(resultados['distances'][0][i], 3))
    print(resultados['documents'][0][i][:200])
    print('---')

faq_taller.md 0.088
domicilio?
Sí, dentro de Valladolid capital, por 25 € ida y vuelta.

## ITV

### ¿Cada cuánto tiene que pasar la ITV un turismo?
Según el Real Decreto 920/2017, un turismo de uso particular está exent
---
rd_920_2017_inspeccion_tecnica_vehiculos.html 0.12
.
– Estación: Se consignará el número asignado a la estación ITV.
– Líneas: Se indicará la o las líneas de la estación ITV en las que ha sido realizada la inspección.
– Tipo de inspección: Se hará con
---
rd_920_2017_inspeccion_tecnica_vehiculos.html 0.124
codificación del
anexo II del Reglamento General de Vehículos
. Constará como máximo de cuatro cifras.
– Marca: Se consignará la marca del vehículo.
– Tipo: Se consignará el tipo del vehículo que figu
---


In [12]:
pregunta = '¿Qué hago si mi coche tiene una llamada a revisión?'
vector = model.encode('query: ' + pregunta, normalize_embeddings=True).tolist()
resultados = collection.query(query_embeddings=[vector], n_results=3)

for i in range(3):
    print(resultados['metadatas'][0][i]['fuente'], round(resultados['distances'][0][i], 3))
    print(resultados['documents'][0][i][:200])
    print('---')

dgt_llamadas_a_revision.html 0.102
DGT - Llamadas a revisión de un vehículo
Llamadas a revisión de un vehículo
Tener tu vehículo siempre a punto es un factor clave para una buena seguridad vial.
Fecha actualización:
18 junio 2025
X
Fac
---
dgt_llamadas_a_revision.html 0.107
 de campañas a revisión de vehículos.
Consulta si tu vehículo tiene llamadas a revisión pendientes
Los avisos de llamadas a revisión que hacen los fabricante se notifican tanto a la DGT como a los due
---
dgt_llamadas_a_revision.html 0.114
s accidentes.
En el Registro de Vehículos de la DGT se recogen electrónicamente todas las incidencias que se producen a lo largo de la vida útil de un vehículo, es decir, desde que se matricula hasta 
---


In [13]:
pregunta = '¿Qué estaciones ITV hay en Valladolid?'
vector = model.encode('query: ' + pregunta, normalize_embeddings=True).tolist()
resultados = collection.query(query_embeddings=[vector], n_results=3)

for i in range(3):
    print(resultados['metadatas'][0][i]['fuente'], round(resultados['distances'][0][i], 3))
    print(resultados['documents'][0][i][:200])
    print('---')

registro_estaciones_itv_2026.csv 0.109
n ITV 4202 de GRUPO ITEVELESA, S.L. (Concesión) está en C/ Universidad, 112, BURGO DE OSMA (Soria, CAST Y L).
La estación ITV 4203 de GRUPO ITEVELESA, S.L. (Concesión) está en ESPACIO IND UI-8 PARC 1-
---
registro_estaciones_itv_2026.csv 0.115
stá en UNIDAD MOVIL AGRÍCOLA Y CICLOMOTORES, Castellanos de Moriscos (Salamanca, CAST Y L).
La estación ITV 40A1' de Atisae (40A1) (Concesión) está en UNIDAD MOVIL AGRÍCOLA Y CICLOMOTORES, Valverde de
---
registro_estaciones_itv_2026.csv 0.12
SA, S.L. (Concesión) está en Polígono Industrial “San Isidro”, Fase II, C/B, Parcela B-31, SALAS DE LOS I. (Burgos, CAST Y L).
La estación ITV 908 de ATISAE-CAL (Concesión) está en CTRA BAÑUELOS, K 0,
---


# Observaciones

- Los 850 trozos están guardados en Chroma con su fuente; el trozo más largo ocupa 360 tokens, por debajo del máximo de 512 del modelo, así que no se corta ninguno.
- ITV: el mejor resultado es la FAQ del taller, que responde justo a la pregunta; los otros dos son trozos del Real Decreto 920/2017 (HTML) que hablan de la ITV pero no de los plazos.
- Llamada a revisión: los tres resultados vienen de la página de la DGT, que es la fuente correcta.
- Valladolid: devuelve estaciones de Castilla y León (Soria, Salamanca, Burgos), pero no de Valladolid. El modelo entiende la región, pero no distingue bien el nombre concreto de la provincia.
- En estas pruebas no ha salido el XML, pero tiene el mismo contenido que el HTML (el Real Decreto 920/2017), así que en otras preguntas pueden aparecer trozos repetidos.
- Estos fallos son los que intentará corregir el re-ranking en la versión mejorada.